# Clasificación Donato vs Fondo (Transfer Learning)

Basado en el notebook de clase *Clasificación de Imágenes de Perros y Gatos con VGG-16* 

Etiquetas:
- **Label 1:** `Donato`
- **Label 2:** `Fondo` (imágenes sin mi rostro: fondos y caras de otras personas)

## 1. Librerías

In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

## 2. Configuración

In [ ]:
NOMBRE = 'Donato'
CLASES = [NOMBRE, 'Fondo']   # índice 0 = estudiante, 1 = fondo
TAM = 224
PESOS = 'imagenet'

# Hiperparámetros (se pueden variar para la parte opcional)
EPOCHS = 30
BATCH_SIZE = 32
OPTIMIZER = 'adam'
LOSS = 'categorical_crossentropy'

## 3. Carga de imágenes

Las fotos se capturan con `1_capturar_fotos.py` (carpeta `dataset/Donato`) y la clase Fondo se llena con `2_descargar_fondos.py` más fondos de Google Images (carpeta `dataset/Fondo`).

Se convierte BGR→RGB y se aplica `preprocess_input` de VGG16, que es la normalización con la que se entrenó la red en ImageNet.

In [ ]:
def preparar(img):
    img = cv2.resize(img, (TAM, TAM))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return preprocess_input(img.astype('float32'))

def load_images(folder, etiqueta):
    images, labels = [], []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            images.append(preparar(img))
            labels.append(etiqueta)
    return images, labels

est_images, est_labels = load_images(f'dataset/{NOMBRE}', 0)
fondo_images, fondo_labels = load_images('dataset/Fondo', 1)
print(f'{NOMBRE}: {len(est_images)} | Fondo: {len(fondo_images)}')

images = np.array(est_images + fondo_images)
labels = to_categorical(np.array(est_labels + fondo_labels), num_classes=2)

## 4. División entrenamiento / prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels.argmax(1))
print(X_train.shape, X_test.shape)

## 5. Modelo VGG-16 con transfer learning

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

baseModel = VGG16(weights=PESOS, include_top=False, input_shape=(TAM, TAM, 3))

model = Sequential()
model.add(baseModel)
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(2, activation='softmax'))

for layer in baseModel.layers:
    layer.trainable = False

model.compile(loss=LOSS, optimizer=OPTIMIZER, metrics=['accuracy'])

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stopping])

loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss: {loss:.4f}')
print(f'Accuracy: {accuracy:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.show()

## 6. Guardar el modelo (lo usa la app de Streamlit)

In [ ]:
model.save('modelo_vgg16.keras')
with open('clases.json', 'w', encoding='utf-8') as f:
    json.dump(CLASES, f, ensure_ascii=False)

## 7. Pruebas con imágenes nunca vistas

Coloca las imágenes en la carpeta `pruebas/`.

In [ ]:
def predecir(ruta):
    img = cv2.imread(ruta)
    pred = model.predict(np.expand_dims(preparar(img), 0), verbose=0)[0]
    etiqueta = CLASES[int(np.argmax(pred))]
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'{etiqueta}  ({pred.max():.1%})')
    plt.axis('off')
    plt.show()
    return etiqueta, pred

for f in sorted(os.listdir('pruebas')):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        print(f, predecir(os.path.join('pruebas', f)))

## 8. Hyperparameter testing (opcional)

Cambia los valores de la sección 2 y vuelve a correr desde la sección 5. Registro de resultados:

| Prueba | Imágenes (G/F) | Loss | Optimizer | Epochs | Batch | Val accuracy |
|---|---|---|---|---|---|---|
| Base | | categorical_crossentropy | adam | 30 | 32 | |
| 2 | | | sgd | | | |
| 3 | | | | 50 | 16 | |